# Transformer — Multivariate Time-Series Forecasting

## Objective

In this stage, a Transformer-based model is developed for next-day sales forecasting.

The model uses the previous 28 days of historical information and 22 engineered features to predict the next day's sales.

## Model Architecture

```text
28 Days × 22 Features
          ↓
   Input Projection
          ↓
 Positional Encoding
          ↓
 Transformer Encoder
          ↓
Global Average Pooling
          ↓
      Dense Layer
          ↓
     Output Layer
          ↓
   Next-Day Sales

In [3]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tensorflow as tf

SEQUENCE_LENGTH = 28
FORECAST_HORIZON = 1
TARGET_COLUMN = "sales"

FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

DATA_PATH = "/kaggle/input/datasets/gou14226/m5-features-event-snap/features_event_snap.parquet"

TRAIN_END_DATE = "2016-03-27"
VALIDATION_START_DATE = "2016-03-28"
VALIDATION_END_DATE = "2016-04-24"

BATCH_SIZE = 64
EPOCHS = 10

TRAIN_STEPS = 5000
VALIDATION_STEPS = 500

print("Configuration loaded successfully.")
print("Sequence length:", SEQUENCE_LENGTH)
print("Forecast horizon:", FORECAST_HORIZON)
print("Number of features:", len(FEATURE_COLUMNS))
print("Batch size:", BATCH_SIZE)

Configuration loaded successfully.
Sequence length: 28
Forecast horizon: 1
Number of features: 22
Batch size: 64


In [4]:
# Verify dataset

print("Dataset exists:", os.path.exists(DATA_PATH))

pf = pq.ParquetFile(DATA_PATH)

print("Rows:", pf.metadata.num_rows)
print("Columns:", pf.metadata.num_columns)
print("Row groups:", pf.num_row_groups)

print("\nColumns:")
print(pf.schema.names)

Dataset exists: True
Rows: 58327370
Columns: 42
Row groups: 61

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'price_change_1', 'price_change_pct_1', 'price_relative_7', 'is_event_day', 'event_count', 'snap_active']


In [5]:
# Verify date range and train-validation split

first_date = None
last_date = None

for rg_idx in range(pf.num_row_groups):
    df_rg = pf.read_row_group(rg_idx, columns=["date"])

    dates = pd.to_datetime(df_rg["date"])

    rg_first = dates.min()
    rg_last = dates.max()

    if first_date is None or rg_first < first_date:
        first_date = rg_first

    if last_date is None or rg_last > last_date:
        last_date = rg_last

    del df_rg

print("Global date range:")
print("Start:", first_date.date())
print("End  :", last_date.date())

print("\nTrain end date:")
print(TRAIN_END_DATE)

print("\nValidation period:")
print(VALIDATION_START_DATE, "to", VALIDATION_END_DATE)

Global date range:
Start: 2011-01-29
End  : 2016-04-24

Train end date:
2016-03-27

Validation period:
2016-03-28 to 2016-04-24


In [6]:
# Feature preparation function

def prepare_features(df, feature_columns):
    df = df.copy()

    # Price-related features:
    # Missing price means price was unavailable.
    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    for col in price_features:
        df[col] = df[col].fillna(0)

    # Historical and rolling features
    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    for col in history_features:
        df[col] = df[col].fillna(0)

    return df[feature_columns]

In [7]:
# Test feature preparation on one row group

test_table = pf.read_row_group(
    0,
    columns=["date"] + FEATURE_COLUMNS
)

test_df = test_table.to_pandas()

test_features = prepare_features(
    test_df,
    FEATURE_COLUMNS
)

print("Original shape:", test_df.shape)
print("Prepared shape:", test_features.shape)

print("Expected feature count:", len(FEATURE_COLUMNS))
print("Actual feature count:", test_features.shape[1])

print("\nNaN values:", test_features.isna().sum().sum())
print("Inf values:", np.isinf(test_features.to_numpy()).sum())

del test_table, test_df, test_features

Original shape: (1048576, 23)
Prepared shape: (1048576, 22)
Expected feature count: 22
Actual feature count: 22

NaN values: 0
Inf values: 0


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_rows = 0

for rg_idx in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg_idx,
        columns=["date"] + FEATURE_COLUMNS
    )

    # Convert PyArrow Table to Pandas DataFrame
    df_rg = table.to_pandas()

    df_rg["date"] = pd.to_datetime(df_rg["date"])

    # Only training rows
    train_df = df_rg[
        df_rg["date"] <= TRAIN_END_DATE
    ]

    if len(train_df) > 0:

        X_train = prepare_features(
            train_df,
            FEATURE_COLUMNS
        ).to_numpy(dtype=np.float64)

        scaler.partial_fit(X_train)

        train_rows += len(train_df)

    del table, df_rg, train_df

print("Training rows used for scaler:", train_rows)
print("Number of features:", len(FEATURE_COLUMNS))

print("\nFirst 5 feature means:")
print(scaler.mean_[:5])

print("\nFirst 5 feature standard deviations:")
print(scaler.scale_[:5])

Training rows used for scaler: 57473650
Number of features: 22

First 5 feature means:
[ 1.12245843  3.4636664   0.7859991  15.71458886 26.0397878 ]

First 5 feature standard deviations:
[ 3.87698193  3.51547202  0.41012744  8.79354866 15.17225613]


In [10]:
# Create sequences for a single item-store series

def create_sequences(
    df,
    feature_columns,
    scaler,
    sequence_length=28,
    target_column="sales"
):
    df = df.sort_values("date").reset_index(drop=True)

    X_raw = prepare_features(
        df,
        feature_columns
    ).to_numpy(dtype=np.float64)

    X_scaled = scaler.transform(X_raw).astype(np.float32)

    y = df[target_column].to_numpy(dtype=np.float32)

    X_sequences = []
    y_targets = []

    for i in range(sequence_length, len(df)):
        X_sequences.append(
            X_scaled[i-sequence_length:i]
        )
        y_targets.append(y[i])

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_targets, dtype=np.float32)
    )

In [11]:
# Test sequence generation on one complete item-store series

series_item = "HOBBIES_1_001"
series_store = "CA_1"

series_parts = []

for rg_idx in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg_idx,
        columns=[
            "item_id",
            "store_id",
            "date"
        ] + FEATURE_COLUMNS
    )

    df_rg = table.to_pandas()

    mask = (
        (df_rg["item_id"] == series_item) &
        (df_rg["store_id"] == series_store)
    )

    if mask.any():
        series_parts.append(
            df_rg.loc[mask].copy()
        )

    del table, df_rg

series_df = pd.concat(
    series_parts,
    ignore_index=True
)

series_df["date"] = pd.to_datetime(series_df["date"])

series_df = series_df.sort_values("date").reset_index(drop=True)

X_test, y_test = create_sequences(
    series_df,
    FEATURE_COLUMNS,
    scaler,
    sequence_length=SEQUENCE_LENGTH,
    target_column=TARGET_COLUMN
)

print("Series:", series_item, "/", series_store)
print("Rows:", len(series_df))
print("Date range:", series_df["date"].min().date(),
      "to", series_df["date"].max().date())

print("\nX shape:", X_test.shape)
print("y shape:", y_test.shape)

print("\nExpected X shape:")
print(f"({len(series_df) - SEQUENCE_LENGTH}, "
      f"{SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})")

print("\nData type:", X_test.dtype)

print("NaN in X:", np.isnan(X_test).sum())
print("Inf in X:", np.isinf(X_test).sum())

print("\nFirst 10 targets:")
print(y_test[:10])

del series_parts, series_df, X_test, y_test

Series: HOBBIES_1_001 / CA_1
Rows: 1913
Date range: 2011-01-29 to 2016-04-24

X shape: (1885, 28, 22)
y shape: (1885,)

Expected X shape:
(1885, 28, 22)

Data type: float32
NaN in X: 0
Inf in X: 0

First 10 targets:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [12]:
# Memory-safe streaming batch generator

class StreamingBatchGenerator:

    def __init__(
        self,
        data_path,
        feature_columns,
        scaler,
        sequence_length,
        target_column,
        start_date,
        end_date,
        batch_size=64
    ):
        self.data_path = data_path
        self.feature_columns = feature_columns
        self.scaler = scaler
        self.sequence_length = sequence_length
        self.target_column = target_column
        self.start_date = pd.Timestamp(start_date)
        self.end_date = pd.Timestamp(end_date)
        self.batch_size = batch_size

    def batches(self):

        pf = pq.ParquetFile(self.data_path)

        history_buffer = {}

        X_batch = []
        y_batch = []

        columns = [
            "item_id",
            "store_id",
            "date"
        ] + self.feature_columns

        for rg_idx in range(pf.num_row_groups):

            table = pf.read_row_group(
                rg_idx,
                columns=columns
            )

            df_rg = table.to_pandas()

            df_rg["date"] = pd.to_datetime(
                df_rg["date"]
            )

            df_rg = df_rg.sort_values(
                ["item_id", "store_id", "date"]
            )

            for (item_id, store_id), group in df_rg.groupby(
                ["item_id", "store_id"],
                sort=False
            ):

                key = (item_id, store_id)

                group = group.reset_index(drop=True)

                # Add previous history from previous row group
                if key in history_buffer:

                    group = pd.concat(
                        [
                            history_buffer[key],
                            group
                        ],
                        ignore_index=True
                    )

                X_raw = prepare_features(
                    group,
                    self.feature_columns
                ).to_numpy(dtype=np.float64)

                X_scaled = self.scaler.transform(
                    X_raw
                ).astype(np.float32)

                y = group[
                    self.target_column
                ].to_numpy(dtype=np.float32)

                dates = group["date"].to_numpy()

                for i in range(
                    self.sequence_length,
                    len(group)
                ):

                    target_date = pd.Timestamp(
                        dates[i]
                    )

                    if (
                        self.start_date
                        <= target_date
                        <= self.end_date
                    ):

                        X_batch.append(
                            X_scaled[
                                i-self.sequence_length:i
                            ]
                        )

                        y_batch.append(
                            y[i]
                        )

                        if len(X_batch) == self.batch_size:

                            yield (
                                np.asarray(
                                    X_batch,
                                    dtype=np.float32
                                ),
                                np.asarray(
                                    y_batch,
                                    dtype=np.float32
                                )
                            )

                            X_batch = []
                            y_batch = []

                # Keep last 28 rows for continuity
                history_buffer[key] = group.tail(
                    self.sequence_length
                ).copy()

            del table, df_rg

        # Yield final partial batch
        if X_batch:

            yield (
                np.asarray(
                    X_batch,
                    dtype=np.float32
                ),
                np.asarray(
                    y_batch,
                    dtype=np.float32
                )
            )

In [13]:
# Test training batch generator

train_generator = StreamingBatchGenerator(
    data_path=DATA_PATH,
    feature_columns=FEATURE_COLUMNS,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    target_column=TARGET_COLUMN,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    batch_size=BATCH_SIZE
)

X_train_test, y_train_test = next(
    train_generator.batches()
)

print("Training batch X shape:", X_train_test.shape)
print("Training batch y shape:", y_train_test.shape)

print("X dtype:", X_train_test.dtype)
print("y dtype:", y_train_test.dtype)

print("NaN in X:", np.isnan(X_train_test).sum())
print("Inf in X:", np.isinf(X_train_test).sum())

print("\nFirst 10 training targets:")
print(y_train_test[:10])

Training batch X shape: (64, 28, 22)
Training batch y shape: (64,)
X dtype: float32
y dtype: float32
NaN in X: 0
Inf in X: 0

First 10 training targets:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [14]:
# Test validation batch generator

validation_generator = StreamingBatchGenerator(
    data_path=DATA_PATH,
    feature_columns=FEATURE_COLUMNS,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    target_column=TARGET_COLUMN,
    start_date=VALIDATION_START_DATE,
    end_date=VALIDATION_END_DATE,
    batch_size=BATCH_SIZE
)

X_val_test, y_val_test = next(
    validation_generator.batches()
)

print("Validation batch X shape:", X_val_test.shape)
print("Validation batch y shape:", y_val_test.shape)

print("X dtype:", X_val_test.dtype)
print("y dtype:", y_val_test.dtype)

print("NaN in X:", np.isnan(X_val_test).sum())
print("Inf in X:", np.isinf(X_val_test).sum())

print("\nFirst 10 validation targets:")
print(y_val_test[:10])

Validation batch X shape: (64, 28, 22)
Validation batch y shape: (64,)
X dtype: float32
y dtype: float32
NaN in X: 0
Inf in X: 0

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [15]:
# TensorFlow dataset generators

def train_batch_stream():
    generator = StreamingBatchGenerator(
        data_path=DATA_PATH,
        feature_columns=FEATURE_COLUMNS,
        scaler=scaler,
        sequence_length=SEQUENCE_LENGTH,
        target_column=TARGET_COLUMN,
        start_date="2011-01-29",
        end_date=TRAIN_END_DATE,
        batch_size=BATCH_SIZE
    )

    yield from generator.batches()


def validation_batch_stream():
    generator = StreamingBatchGenerator(
        data_path=DATA_PATH,
        feature_columns=FEATURE_COLUMNS,
        scaler=scaler,
        sequence_length=SEQUENCE_LENGTH,
        target_column=TARGET_COLUMN,
        start_date=VALIDATION_START_DATE,
        end_date=VALIDATION_END_DATE,
        batch_size=BATCH_SIZE
    )

    yield from generator.batches()


output_signature = (
    tf.TensorSpec(
        shape=(None, SEQUENCE_LENGTH, len(FEATURE_COLUMNS)),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(None,),
        dtype=tf.float32
    )
)

train_dataset = tf.data.Dataset.from_generator(
    train_batch_stream,
    output_signature=output_signature
)

validation_dataset = tf.data.Dataset.from_generator(
    validation_batch_stream,
    output_signature=output_signature
)

print("TensorFlow datasets created successfully.")

TensorFlow datasets created successfully.


I0000 00:00:1789550633.823195      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789550633.826242      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [16]:
# Test TensorFlow datasets

X_train_tf, y_train_tf = next(iter(train_dataset))

X_val_tf, y_val_tf = next(iter(validation_dataset))

print("Training batch:")
print("X shape:", X_train_tf.shape)
print("y shape:", y_train_tf.shape)
print("X dtype:", X_train_tf.dtype)
print("y dtype:", y_train_tf.dtype)

print("\nValidation batch:")
print("X shape:", X_val_tf.shape)
print("y shape:", y_val_tf.shape)
print("X dtype:", X_val_tf.dtype)
print("y dtype:", y_val_tf.dtype)

print("\nValidation NaN:", tf.math.count_nonzero(
    tf.math.is_nan(X_val_tf)
).numpy())

print("Validation Inf:", tf.math.count_nonzero(
    tf.math.is_inf(X_val_tf)
).numpy())

print("\nFirst 10 validation targets:")
print(y_val_tf[:10].numpy())

Training batch:
X shape: (64, 28, 22)
y shape: (64,)
X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>

Validation batch:
X shape: (64, 28, 22)
y shape: (64,)
X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>

Validation NaN: 0
Validation Inf: 0

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [17]:
# Positional encoding layer

class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, sequence_length, d_model):
        super().__init__()

        positions = np.arange(sequence_length)[:, np.newaxis]
        dimensions = np.arange(d_model)[np.newaxis, :]

        angle_rates = 1 / np.power(
            10000,
            (2 * (dimensions // 2)) / np.float32(d_model)
        )

        angle_rads = positions * angle_rates

        pos_encoding = np.zeros(
            (sequence_length, d_model)
        )

        pos_encoding[:, 0::2] = np.sin(
            angle_rads[:, 0::2]
        )

        pos_encoding[:, 1::2] = np.cos(
            angle_rads[:, 1::2]
        )

        self.pos_encoding = tf.cast(
            pos_encoding[np.newaxis, ...],
            dtype=tf.float32
        )

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

In [18]:
# Transformer encoder block

class TransformerEncoder(tf.keras.layers.Layer):

    def __init__(
        self,
        d_model,
        num_heads,
        ff_dim,
        dropout=0.1
    ):
        super().__init__()

        self.attention = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(
                ff_dim,
                activation="relu"
            ),
            tf.keras.layers.Dense(d_model)
        ])

        self.layernorm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.layernorm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, inputs, training=False):

        attention_output = self.attention(
            inputs,
            inputs,
            training=training
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(
            out1
        )

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        return self.layernorm2(
            out1 + ffn_output
        )

In [19]:
# Transformer model configuration

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

inputs = tf.keras.Input(
    shape=(
        SEQUENCE_LENGTH,
        len(FEATURE_COLUMNS)
    )
)

# Project 22 input features into 64-dimensional representation
x = tf.keras.layers.Dense(
    D_MODEL
)(inputs)

# Add positional information
x = PositionalEncoding(
    SEQUENCE_LENGTH,
    D_MODEL
)(x)

# Transformer encoder
x = TransformerEncoder(
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT
)(x)

# Aggregate information from all 28 time steps
x = tf.keras.layers.GlobalAveragePooling1D()(x)

# Prediction head
x = tf.keras.layers.Dense(
    32,
    activation="relu"
)(x)

outputs = tf.keras.layers.Dense(1)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 22)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 28, 64)         │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 28, 64)         │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, 28, 64)         │        33,472 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,057 (144.75 KB)

 Trainable params: 37,057 (144.75 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Forward-pass test

sample_output = model(
    X_train_tf,
    training=False
)

print("Input shape:", X_train_tf.shape)
print("Output shape:", sample_output.shape)
print("Output dtype:", sample_output.dtype)

print("\nNaN in output:",
      tf.math.count_nonzero(
          tf.math.is_nan(sample_output)
      ).numpy())

print("Inf in output:",
      tf.math.count_nonzero(
          tf.math.is_inf(sample_output)
      ).numpy())

print("\nFirst 10 predictions:")
print(sample_output[:10].numpy().reshape(-1))

Input shape: (64, 28, 22)
Output shape: (64, 1)
Output dtype: <dtype: 'float32'>

NaN in output: 0
Inf in output: 0

First 10 predictions:
[-0.80991364 -0.8071624  -0.790248   -0.76744294 -0.7617322  -0.7570425
 -0.753253   -0.753161   -0.7579818  -0.74586546]


In [21]:
# Early stopping configuration

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

print("Early stopping configured successfully.")
print("Monitor:", early_stopping.monitor)
print("Patience:", early_stopping.patience)
print("Restore best weights:", early_stopping.restore_best_weights)
print("Maximum epochs:", EPOCHS)
print("Training steps per epoch:", TRAIN_STEPS)
print("Validation steps:", VALIDATION_STEPS)

Early stopping configured successfully.
Monitor: val_loss
Patience: 2
Restore best weights: True
Maximum epochs: 10
Training steps per epoch: 5000
Validation steps: 500


In [22]:
# Train Transformer model

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VALIDATION_STEPS,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
  38/5000 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - loss: 0.2625 - mae: 0.3921

I0000 00:00:1789550854.155687     200 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5000/5000 ━━━━━━━━━━━━━━━━━━━━ 57s 10ms/step - loss: 6.9543 - mae: 0.9317 - val_loss: 4.8040 - val_mae: 1.1857
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 46s 9ms/step - loss: 3.8821 - mae: 0.6010 - val_loss: 4.1368 - val_mae: 0.9378
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 47s 9ms/step - loss: 1.7576 - mae: 0.6760 - val_loss: 4.5408 - val_mae: 1.0859
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 47s 9ms/step - loss: 4.8615 - mae: 0.9975 - val_loss: 4.0414 - val_mae: 0.9844
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 45s 9ms/step - loss: 3.5348 - mae: 0.8377 - val_loss: 4.0210 - val_mae: 1.0298
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 48s 10ms/step - loss: 3.2215 - mae: 1.0204 - val_loss: 3.9977 - val_mae: 0.9803
Epoch 7/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 45s 9ms/step - loss: 3.2255 - mae: 0.8464 - val_loss: 4.1732 - val_mae: 0.9761
Epoch 8/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 46s 9ms/step - loss: 0.9586 - mae: 0.4847 - val_loss: 3.9832 - val_mae: 0.9509
Epoch 9/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━

In [23]:
# Generate predictions for the complete validation period

val_predictions = model.predict(
    validation_dataset,
    verbose=1
).reshape(-1)

print("Prediction shape:", val_predictions.shape)
print("Expected predictions:", 853720)

print("NaN predictions:", np.isnan(val_predictions).sum())
print("Inf predictions:", np.isinf(val_predictions).sum())

print("\nFirst 10 predictions:")
print(val_predictions[:10])

13340/13340 ━━━━━━━━━━━━━━━━━━━━ 407s 30ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Prediction shape: (853720,)
Expected predictions: 853720
NaN predictions: 0
Inf predictions: 0

First 10 predictions:
[0.67348677 0.567449   0.567449   0.567449   0.567449   0.7529433
 0.78171074 0.71716255 0.47433314 0.70971364]


In [24]:
# Collect actual sales values for the complete validation period

val_actuals = []

for X_batch, y_batch in validation_dataset:
    val_actuals.append(y_batch.numpy())

val_actuals = np.concatenate(val_actuals)

print("Actual shape:", val_actuals.shape)
print("Expected actuals:", 853720)

print("NaN actuals:", np.isnan(val_actuals).sum())
print("Inf actuals:", np.isinf(val_actuals).sum())

print("\nFirst 10 actual values:")
print(val_actuals[:10])

Actual shape: (853720,)
Expected actuals: 853720
NaN actuals: 0
Inf actuals: 0

First 10 actual values:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(val_actuals, val_predictions)

rmse = np.sqrt(
    mean_squared_error(val_actuals, val_predictions)
)

wape = (
    np.sum(np.abs(val_actuals - val_predictions))
    / np.sum(np.abs(val_actuals))
) * 100

print("Transformer Validation Metrics")
print("=" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.4f}%")

Transformer Validation Metrics
MAE  : 1.0217
RMSE : 2.5463
WAPE : 73.6934%


# Transformer Model — Final Results

## Model Configuration

- Sequence Length: 28 days
- Forecast Horizon: 1 day
- Input Features: 22
- Target: `sales`
- Architecture:
  - Input Projection: Dense(64)
  - Positional Encoding
  - Transformer Encoder
  - 4 Attention Heads
  - Feed-Forward Dimension: 128
  - Dropout: 0.1
  - Global Average Pooling
  - Dense(32)
  - Output: Dense(1)
- Optimizer: Adam
- Learning Rate: 0.001
- Loss: MSE
- Training Batch Size: 64
- Maximum Epochs: 10
- Training Steps per Epoch: 5000
- Validation Steps per Epoch: 500

## Training Results

- Total Epochs Completed: 10
- Best Epoch: 10
- Best Validation Loss: 3.9079
- Best Validation MAE: 0.9554
- Best weights were retained based on validation loss.

## Validation Results

Validation Period:

2016-03-28 to 2016-04-24

Total validation predictions:

853,720

| Metric | Transformer |
|---|---:|
| MAE | 1.0217 |
| RMSE | 2.5463 |
| WAPE | 73.6934% |

Prediction verification:

- Predictions: 853,720
- Actual values: 853,720
- NaN predictions: 0
- Inf predictions: 0
- NaN actuals: 0
- Inf actuals: 0

## Model Comparison

| Model | MAE | RMSE | WAPE |
|---|---:|---:|---:|
| Naive | 1.1798 | 2.5948 | 85.0972% |
| Seasonal Naive | 1.2054 | 2.6601 | 86.9390% |
| 28-day Moving Average | 1.0050 | 2.0935 | 72.4875% |
| RNN | 1.0326 | 2.4927 | 74.4774% |
| LSTM | 1.0220 | 2.3766 | 73.7131% |
| GRU | 1.0691 | 2.5656 | 77.1137% |
| LSTM + Attention | 1.0058 | 2.2867 | 72.5478% |
| Transformer | 1.0217 | 2.5463 | 73.6934% |

## Conclusion

The Transformer successfully learned the multivariate time-series forecasting task and produced valid predictions for the complete validation period.

It improved over the Naive and Seasonal Naive baselines and achieved performance comparable to the recurrent deep learning models.

The Transformer achieved:

- MAE: 1.0217
- RMSE: 2.5463
- WAPE: 73.6934%

The 28-day Moving Average and LSTM + Attention models achieved lower validation errors in this experiment.

Therefore, the Transformer experiment provides a useful comparison point for evaluating recurrent and attention-based architectures in the later model comparison stage.

## Stage Status

**Phase 9 — Transformer: COMPLETE**